# 04 - Triple Model Comparison

Step 4 compares Logistic Regression, Random Forest, and XGBoost using PR-AUC under class imbalance.
All outputs are saved to `step4_modeling/` directories.


In [ ]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
# ---------- Global Paths ----------
# Reproducible root: run notebook from project root directory.
# If needed, replace os.getcwd() with your explicit coursework folder path.
PROJECT_ROOT = os.getcwd()

# Keep original six-step directories
STEP_DIRS = {
    "step1": os.path.join(PROJECT_ROOT, "step1_obtain_frame"),
    "step2": os.path.join(PROJECT_ROOT, "step2_eda"),
    "step3": os.path.join(PROJECT_ROOT, "step3_prepare"),
    "step4": os.path.join(PROJECT_ROOT, "step4_shortlist"),
    "step5": os.path.join(PROJECT_ROOT, "step5_finetune"),
    "step6": os.path.join(PROJECT_ROOT, "step6_present"),
}

for step_dir in STEP_DIRS.values():
    os.makedirs(os.path.join(step_dir, "plots"), exist_ok=True)
    os.makedirs(os.path.join(step_dir, "tables"), exist_ok=True)

# Dedicated modeling directory requested for Step 4 outputs
STEP4_MODELING_DIR = os.path.join(PROJECT_ROOT, "step4_modeling")
STEP4_MODELING_PLOT_DIR = os.path.join(STEP4_MODELING_DIR, "plots")
STEP4_MODELING_TABLE_DIR = os.path.join(STEP4_MODELING_DIR, "tables")
STEP4_MODELING_MODEL_DIR = os.path.join(STEP4_MODELING_DIR, "models")

os.makedirs(STEP4_MODELING_PLOT_DIR, exist_ok=True)
os.makedirs(STEP4_MODELING_TABLE_DIR, exist_ok=True)
os.makedirs(STEP4_MODELING_MODEL_DIR, exist_ok=True)

# Step 3 prepared inputs
STEP3_TABLE_DIR = os.path.join(STEP_DIRS["step3"], "tables")
X_TRAIN_PATH = os.path.join(STEP3_TABLE_DIR, "X_train_processed.pkl")
X_TEST_PATH = os.path.join(STEP3_TABLE_DIR, "X_test_processed.pkl")
Y_TRAIN_PATH = os.path.join(STEP3_TABLE_DIR, "y_train.pkl")
Y_TEST_PATH = os.path.join(STEP3_TABLE_DIR, "y_test.pkl")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Step4 modeling plot dir:", STEP4_MODELING_PLOT_DIR)
print("Step4 modeling table dir:", STEP4_MODELING_TABLE_DIR)
print("Step4 modeling model dir:", STEP4_MODELING_MODEL_DIR)


In [ ]:
# ---------- Utility Functions ----------
def load_pickle(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)


def save_pickle(obj, path: str) -> None:
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"Saved model: {path}")


def save_table(df: pd.DataFrame, filename: str) -> str:
    path = os.path.join(STEP4_MODELING_TABLE_DIR, filename)
    df.to_csv(path, index=False)
    print(f"Saved table: {path}")
    return path


def save_plot(filename: str, dpi: int = 300) -> str:
    path = os.path.join(STEP4_MODELING_PLOT_DIR, filename)
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"Saved plot: {path}")
    return path


def to_numpy_1d(y):
    if isinstance(y, (pd.Series, pd.DataFrame)):
        y_arr = np.ravel(y.values)
    else:
        y_arr = np.ravel(np.array(y))
    return y_arr.astype(int)

## 1) Load Prepared Data


In [ ]:
X_train = load_pickle(X_TRAIN_PATH)
X_test = load_pickle(X_TEST_PATH)
y_train = to_numpy_1d(load_pickle(Y_TRAIN_PATH))
y_test = to_numpy_1d(load_pickle(Y_TEST_PATH))

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape, "| positive rate:", y_train.mean().round(4))
print("y_test shape:", y_test.shape, "| positive rate:", y_test.mean().round(4))

if not isinstance(X_train, pd.DataFrame):
    X_train = pd.DataFrame(X_train)
if not isinstance(X_test, pd.DataFrame):
    X_test = pd.DataFrame(X_test)

## 2) Model Definitions and 5-Fold Stratified CV (PR-AUC)


In [ ]:
def build_models(y_train_array: np.ndarray):
    """
    Define model configurations requested for class-imbalanced learning.
    Metric objective is PR-AUC (average precision).
    """
    pos_count = (y_train_array == 1).sum()
    neg_count = (y_train_array == 0).sum()
    scale_pos_weight = neg_count / max(pos_count, 1)

    models = {
        "Logistic Regression": LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42,
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=100,
            scale_pos_weight=scale_pos_weight,
            eval_metric="aucpr",
            random_state=42,
            n_jobs=-1,
            objective="binary:logistic",
        ),
    }
    return models, scale_pos_weight


def evaluate_cv(models: dict, X: pd.DataFrame, y: np.ndarray) -> pd.DataFrame:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    rows = []

    for name, model in models.items():
        scores = cross_val_score(
            model,
            X,
            y,
            scoring="average_precision",  # PR-AUC
            cv=cv,
            n_jobs=-1,
        )
        rows.append({
            "model": name,
            "cv_pr_auc_mean": float(scores.mean()),
            "cv_pr_auc_std": float(scores.std()),
        })

    cv_table = pd.DataFrame(rows).sort_values("cv_pr_auc_mean", ascending=False).reset_index(drop=True)
    return cv_table

## 3) Train Models, Plot Test PR Curves, and Compare


In [ ]:
def fit_models(models: dict, X_train_df: pd.DataFrame, y_train_arr: np.ndarray) -> dict:
    fitted = {}
    for name, model in models.items():
        model.fit(X_train_df, y_train_arr)
        fitted[name] = model
    return fitted


def plot_pr_curves(models_fitted: dict, X_test_df: pd.DataFrame, y_test_arr: np.ndarray) -> pd.DataFrame:
    rows = []
    plt.figure(figsize=(9, 7))

    for name, model in models_fitted.items():
        y_score = model.predict_proba(X_test_df)[:, 1]
        precision, recall, _ = precision_recall_curve(y_test_arr, y_score)
        ap = average_precision_score(y_test_arr, y_score)

        rows.append({"model": name, "test_pr_auc": float(ap)})
        plt.plot(recall, precision, linewidth=2, label=f"{name} (AP={ap:.3f})")

    baseline = y_test_arr.mean()
    plt.hlines(baseline, xmin=0, xmax=1, colors="gray", linestyles="--", label=f"Baseline={baseline:.3f}")
    plt.title("Precision-Recall Curve Comparison (Test Set)")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.legend(loc="best")
    plt.tight_layout()
    save_plot("41_test_precision_recall_comparison.png")
    plt.show()

    test_table = pd.DataFrame(rows).sort_values("test_pr_auc", ascending=False).reset_index(drop=True)
    return test_table

## 4) Feature Importance for Best Ensemble Model (RF vs XGBoost)


In [ ]:
def plot_best_ensemble_feature_importance(models_fitted: dict, test_pr_table: pd.DataFrame, X_train_df: pd.DataFrame, top_n: int = 20):
    ensemble_candidates = ["Random Forest", "XGBoost"]
    ensemble_scores = test_pr_table[test_pr_table["model"].isin(ensemble_candidates)].copy()

    best_model_name = ensemble_scores.sort_values("test_pr_auc", ascending=False).iloc[0]["model"]
    best_model = models_fitted[best_model_name]

    if not hasattr(best_model, "feature_importances_"):
        print(f"Model {best_model_name} does not expose feature_importances_.")
        return None

    importances = pd.DataFrame({
        "feature": X_train_df.columns,
        "importance": best_model.feature_importances_,
    }).sort_values("importance", ascending=False)

    top_imp = importances.head(top_n).sort_values("importance", ascending=True)

    plt.figure(figsize=(10, 7))
    plt.barh(top_imp["feature"], top_imp["importance"], color="#457b9d")
    plt.title(f"Top {top_n} Feature Importances - {best_model_name}")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    save_plot("42_feature_importance_best_ensemble.png")
    plt.show()

    save_table(importances, "42_feature_importance_best_ensemble_full.csv")

    return {
        "best_ensemble_model": best_model_name,
        "importances": importances,
    }

## 5) Run Triple Model Comparison and Save Outputs


In [ ]:
# Build model set
models, scale_pos_weight = build_models(y_train)
print(f"Computed XGBoost scale_pos_weight: {scale_pos_weight:.4f}")

# 5-Fold Stratified CV on PR-AUC
cv_results = evaluate_cv(models, X_train, y_train)
print("\nCV PR-AUC Summary:")
display(cv_results)
save_table(cv_results, "40_cv_pr_auc_summary.csv")

# Train each model on full training data
fitted_models = fit_models(models, X_train, y_train)

# Test-set PR curve comparison + AP table
test_results = plot_pr_curves(fitted_models, X_test, y_test)
print("\nTest PR-AUC (Average Precision):")
display(test_results)
save_table(test_results, "41_test_pr_auc_summary.csv")

# Feature importance for best ensemble (RF or XGBoost)
fi_result = plot_best_ensemble_feature_importance(fitted_models, test_results, X_train, top_n=20)
if fi_result is not None:
    print("Best ensemble model:", fi_result["best_ensemble_model"])

# Save all trained models
save_pickle(fitted_models["Logistic Regression"], os.path.join(STEP4_MODELING_MODEL_DIR, "logistic_regression_model.pkl"))
save_pickle(fitted_models["Random Forest"], os.path.join(STEP4_MODELING_MODEL_DIR, "random_forest_model.pkl"))
save_pickle(fitted_models["XGBoost"], os.path.join(STEP4_MODELING_MODEL_DIR, "xgboost_model.pkl"))

print("\nStep 4 outputs saved to:")
print("- Models:", STEP4_MODELING_MODEL_DIR)
print("- Plots:", STEP4_MODELING_PLOT_DIR)
print("- Tables:", STEP4_MODELING_TABLE_DIR)
